# NEXUS Football Predictor v2: 1X2 + BTTS across Europe

Gives **home / draw / away** probabilities and **both teams to score** for European leagues (and the Champions League if you add a free API key).

**How the big-team vs small-team rule works:** every team gets a rating (Elo). When a match is predicted, each side's recent form is read only from its past games against opponents of the *same strength band* (weaker / level / stronger than itself). So a big club's results against lower teams are used for games against lower teams, and its results against other big clubs are used for those games.

Run cells 1 to 5 in order. Keep the data limits in mind: football-data.co.uk has no Champions League, and the model is a ranking tool, not proof of an edge over bookmakers.

In [ ]:
# CELL 1: settings
import io, re, os, json, time, difflib, unicodedata
import numpy as np, pandas as pd, requests, joblib
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import log_loss, accuracy_score, roc_auc_score

URL = "https://www.football-data.co.uk/"
MAIN = {"E0": "Premier League", "E1": "Championship", "E2": "League One", "E3": "League Two", "EC": "National League",
        "SC0": "Scottish Premiership", "SC1": "Scottish Championship", "SC2": "Scottish League One", "SC3": "Scottish League Two",
        "D1": "Bundesliga", "D2": "2. Bundesliga", "I1": "Serie A", "I2": "Serie B", "SP1": "La Liga", "SP2": "La Liga 2",
        "F1": "Ligue 1", "F2": "Ligue 2", "N1": "Eredivisie", "B1": "Belgian Pro League", "P1": "Primeira Liga",
        "T1": "Super Lig", "G1": "Greek Super League"}
EXTRA = {"AUT": "Austria", "DNK": "Denmark", "FIN": "Finland", "IRL": "Ireland", "NOR": "Norway",
         "POL": "Poland", "ROU": "Romania", "RUS": "Russia", "SWE": "Sweden", "SWZ": "Switzerland"}
EURO_COUNTRIES = list(EXTRA.values())
SEASONS = ["2122", "2223", "2324", "2425", "2526", "2627"]
WARMUP_END = "2022-07-01"      # the first season only builds ratings, it is not used for training
FD_ORG_TOKEN = ""              # optional: free key from football-data.org adds the Champions League
LEAGUE_START = {"Premier League": 1580, "La Liga": 1570, "Serie A": 1560, "Bundesliga": 1560, "Ligue 1": 1520,
                "Eredivisie": 1490, "Primeira Liga": 1490, "Super Lig": 1470, "Belgian Pro League": 1470,
                "Championship": 1470, "Scottish Premiership": 1440, "Greek Super League": 1430,
                "2. Bundesliga": 1440, "Serie B": 1420, "La Liga 2": 1420, "Ligue 2": 1410, "Champions League": 1500}

SAVE_DIR = "/content"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/BTTS_Project_Backup"
    os.makedirs(SAVE_DIR, exist_ok=True)
except Exception as e:
    print("Drive not mounted, saving locally:", str(e)[:60])
print("Saving to:", SAVE_DIR)

In [ ]:
# CELL 2: helpers (cleaning, ratings, features)
def fetch(url):
    r = requests.get(url, timeout=30); r.raise_for_status()
    d = pd.read_csv(io.BytesIO(r.content), encoding="utf-8-sig", encoding_errors="replace")
    d.columns = [re.sub(r"^[^A-Za-z0-9]+", "", str(c)).strip() for c in d.columns]
    return d

def pick(d, names):
    out = pd.Series(np.nan, index=d.index)
    for n in names:
        if n in d.columns: out = out.combine_first(pd.to_numeric(d[n], errors="coerce"))
    return out

ODDS = {"oH": ["AvgH", "B365H", "PSH"], "oD": ["AvgD", "B365D", "PSD"], "oA": ["AvgA", "B365A", "PSA"],
        "oO": ["Avg>2.5", "B365>2.5", "P>2.5", "Max>2.5"], "oU": ["Avg<2.5", "B365<2.5", "P<2.5", "Max<2.5"]}
ODDS_X = {"oH": ["AvgH", "PH", "MaxH"], "oD": ["AvgD", "PD", "MaxD"], "oA": ["AvgA", "PA", "MaxA"], "oO": [], "oU": []}

def clean_main(d, names):
    d = d.dropna(subset=["HomeTeam", "AwayTeam"])
    out = pd.DataFrame({"date": pd.to_datetime(d["Date"], dayfirst=True, format="mixed", errors="coerce"),
                        "league": d["Div"].map(names), "home": d["HomeTeam"], "away": d["AwayTeam"]})
    for k, v in ODDS.items(): out[k] = pick(d, v)
    if "FTHG" in d.columns:
        out["hg"] = pd.to_numeric(d["FTHG"], errors="coerce"); out["ag"] = pd.to_numeric(d["FTAG"], errors="coerce")
    return out

def clean_extra(d, countries=None):
    d = d.dropna(subset=["Home", "Away"])
    if countries is not None: d = d[d["Country"].isin(countries)]
    out = pd.DataFrame({"date": pd.to_datetime(d["Date"], dayfirst=True, format="mixed", errors="coerce"),
                        "league": d["League"], "home": d["Home"], "away": d["Away"]})
    for k, v in ODDS_X.items(): out[k] = pick(d, v)
    if "HG" in d.columns:
        out["hg"] = pd.to_numeric(d["HG"], errors="coerce"); out["ag"] = pd.to_numeric(d["AG"], errors="coerce")
    return out

# ---- ratings: Elo with goal-difference weighting ----
HOME_ADV, K = 60, 20
ELO, TEAM_HIST = {}, {}
def exp_home(rh, ra): return 1 / (1 + 10 ** (-(rh + HOME_ADV - ra) / 400))
def bucket(gap): return 0 if gap < -100 else (2 if gap > 100 else 1)   # 0 weaker opp, 1 level, 2 stronger opp
def shr(v, prior, k=4): return (sum(v) + k * prior) / (len(v) + k)       # shrink small samples to a prior
def get_elo(t, lg): return ELO.setdefault(t, LEAGUE_START.get(lg, 1400))

def team_block(t, gap, pre):
    H = TEAM_HIST.get(t, []); Hb = [x for x in H if x[3] == bucket(gap)][-8:]; H10 = H[-10:]
    return {pre + "ppg_b": shr([x[2] for x in Hb], 1.35), pre + "gd_b": shr([x[0] - x[1] for x in Hb], 0.0),
            pre + "n_b": len(Hb), pre + "ppg": shr([x[2] for x in H10], 1.35),
            pre + "gf": shr([x[0] for x in H10], 1.35), pre + "ga": shr([x[1] for x in H10], 1.35)}

def build(home, away, lg, o):
    rh, ra = get_elo(home, lg), get_elo(away, lg)
    f = {"elo_h": rh, "elo_a": ra, "elo_diff": rh - ra, "lg_start": LEAGUE_START.get(lg, 1400)}
    f.update(team_block(home, ra - rh, "h_")); f.update(team_block(away, rh - ra, "a_"))
    if all(pd.notna(o.get(k)) for k in ("oH", "oD", "oA")):
        iH, iD, iA = 1 / o["oH"], 1 / o["oD"], 1 / o["oA"]; s = iH + iD + iA
        f.update(pH=iH / s, pD=iD / s, pA=iA / s)
    else:
        f.update(pH=np.nan, pD=np.nan, pA=np.nan)
    ok = pd.notna(o.get("oO")) and pd.notna(o.get("oU"))
    f["pO25"] = (1 / o["oO"]) / ((1 / o["oO"]) + (1 / o["oU"])) if ok else np.nan
    return f

def update(home, away, lg, hg, ag):
    rh, ra = get_elo(home, lg), get_elo(away, lg)
    res = 1.0 if hg > ag else (0.5 if hg == ag else 0.0)
    d = K * (np.log(abs(hg - ag) + 1) + 1) * (res - exp_home(rh, ra))
    ph = 3 if hg > ag else int(hg == ag); pa = 3 if ag > hg else int(hg == ag)
    TEAM_HIST.setdefault(home, []).append((hg, ag, ph, bucket(ra - rh)))
    TEAM_HIST.setdefault(away, []).append((ag, hg, pa, bucket(rh - ra)))
    ELO[home], ELO[away] = rh + d, ra - d

def hgb():
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=250, l2_regularization=2.0,
                                          min_samples_leaf=80, early_stopping=True, validation_fraction=0.1,
                                          n_iter_no_change=20, random_state=42)
print("helpers ready")

In [ ]:
# CELL 3: download history (all listed leagues) + optional Champions League
def load_history():
    parts = []
    for s in SEASONS:
        for c in MAIN:
            try: parts.append(clean_main(fetch(f"{URL}mmz4281/{s}/{c}.csv"), MAIN))
            except Exception as e: print("skip", s, c, str(e)[:40])
    for c in EXTRA:
        try: parts.append(clean_extra(fetch(f"{URL}new/{c}.csv")))
        except Exception as e: print("skip", c, str(e)[:40])
    return pd.concat(parts, ignore_index=True)

ALL = load_history()
ALL = ALL.dropna(subset=["date", "league", "home", "away", "hg", "ag"])
ALL = ALL[ALL["date"] >= "2021-07-01"]
KNOWN = {x.lower(): x for x in ALL["home"].unique()}

# --- Champions League via football-data.org (optional) ---
ALIAS = {"manchester city": "Man City", "manchester united": "Man United", "tottenham hotspur": "Tottenham",
         "newcastle united": "Newcastle", "wolverhampton wanderers": "Wolves", "nottingham forest": "Nott'm Forest",
         "paris saintgermain": "Paris SG", "club atletico de madrid": "Ath Madrid", "atletico madrid": "Ath Madrid",
         "athletic club": "Ath Bilbao", "borussia dortmund": "Dortmund", "bayer 04 leverkusen": "Leverkusen",
         "fc bayern munchen": "Bayern Munich", "bayern munchen": "Bayern Munich", "borussia monchengladbach": "M'gladbach",
         "eintracht frankfurt": "Ein Frankfurt", "internazionale milano": "Inter", "ac milan": "Milan", "as roma": "Roma",
         "sporting clube de portugal": "Sp Lisbon", "sl benfica": "Benfica", "psv": "PSV Eindhoven"}
def canon(n):
    k = unicodedata.normalize("NFKD", n).encode("ascii", "ignore").decode().lower()
    k = re.sub(r"[^a-z ]", "", k)
    k = " ".join(w for w in k.split() if w not in ("fc", "cf", "afc", "fk", "sk", "bk", "ssc", "sc"))
    if k in ALIAS: return ALIAS[k]
    m = difflib.get_close_matches(k, list(KNOWN), 1, 0.75)
    return KNOWN[m[0]] if m else n

def fetch_cl(token):
    rows = []
    for yr in range(2021, 2027):
        try:
            r = requests.get(f"https://api.football-data.org/v4/competitions/CL/matches?season={yr}",
                             headers={"X-Auth-Token": token}, timeout=30); r.raise_for_status()
        except Exception as e:
            print("CL", yr, "skipped:", str(e)[:60]); continue
        for m in r.json().get("matches", []):
            ft = m["score"]["fullTime"]
            rows.append({"date": pd.to_datetime(m["utcDate"]).tz_convert(None).normalize(), "league": "Champions League",
                         "home": canon(m["homeTeam"]["name"]), "away": canon(m["awayTeam"]["name"]),
                         "hg": ft["home"], "ag": ft["away"]})
        time.sleep(7)          # free tier allows 10 requests per minute
    return pd.DataFrame(rows)

CL_UP = pd.DataFrame()
if FD_ORG_TOKEN:
    cl = fetch_cl(FD_ORG_TOKEN)
    if len(cl):
        CL_UP = cl[cl["hg"].isna()]
        ALL = pd.concat([ALL, cl.dropna(subset=["hg", "ag"])], ignore_index=True)
ALL = ALL.sort_values("date").reset_index(drop=True)
print(len(ALL), "matches |", ALL["league"].nunique(), "competitions |", ALL["date"].min().date(), "to", ALL["date"].max().date())

In [ ]:
# CELL 4: build features day by day (no leakage), train, validate, save
WARM = pd.Timestamp(WARMUP_END)
ELO.clear(); TEAM_HIST.clear(); rows = []
for d, day in ALL.groupby("date", sort=True):
    if d >= WARM:                                   # features first...
        for r in day.itertuples():
            f = build(r.home, r.away, r.league, r._asdict())
            f.update(date=d, y=0 if r.hg > r.ag else (1 if r.hg == r.ag else 2), btts=int(r.hg > 0 and r.ag > 0))
            rows.append(f)
    for r in day.itertuples():                      # ...then update ratings and form
        update(r.home, r.away, r.league, int(r.hg), int(r.ag))

feat = pd.DataFrame(rows)
FEATS = [c for c in feat.columns if c not in ("date", "y", "btts")]
ud = np.sort(feat["date"].unique()); cut = ud[int(len(ud) * 0.8)]
tr, te = feat[feat["date"] < cut], feat[feat["date"] >= cut]
print("Train:", len(tr), "| Test:", len(te), "| test from", pd.Timestamp(cut).date())

L = [0, 1, 2]
m1 = hgb().fit(tr[FEATS], tr["y"]); P = m1.predict_proba(te[FEATS])
base = np.tile(np.bincount(tr["y"], minlength=3) / len(tr), (len(te), 1))
print("1X2   log loss  model:", round(log_loss(te["y"], P, labels=L), 4), "| baseline:", round(log_loss(te["y"], base, labels=L), 4),
      "| accuracy:", round(accuracy_score(te["y"], P.argmax(1)), 3))
has = te["pH"].notna().values
if has.sum() > 100:
    bk = te.loc[has, ["pH", "pD", "pA"]].values
    print("On rows with odds   bookmaker:", round(log_loss(te["y"][has], bk, labels=L), 4), "| model:", round(log_loss(te["y"][has], P[has], labels=L), 4))
m2 = hgb().fit(tr[FEATS], tr["btts"]); pb = m2.predict_proba(te[FEATS])[:, 1]
print("BTTS  log loss  model:", round(log_loss(te["btts"], pb), 4), "| baseline:", round(log_loss(te["btts"], np.full(len(pb), tr["btts"].mean())), 4),
      "| AUC:", round(roc_auc_score(te["btts"], pb), 4))

m1f = hgb().fit(feat[FEATS], feat["y"]); m2f = hgb().fit(feat[FEATS], feat["btts"])
joblib.dump({"m1x2": m1f, "mbtts": m2f, "feats": FEATS}, f"{SAVE_DIR}/nexus_models.joblib")
print("Saved models. Ratings now reflect matches up to", ALL["date"].max().date())

In [ ]:
# CELL 5: predict upcoming fixtures (leagues + Champions League if enabled)
parts = []
try: parts.append(clean_main(fetch(URL + "fixtures.csv"), MAIN))
except Exception as e: print("main fixtures failed:", str(e)[:60])
try: parts.append(clean_extra(fetch(URL + "new_league_fixtures.csv"), EURO_COUNTRIES))
except Exception as e: print("extra fixtures failed:", str(e)[:60])
if len(CL_UP): parts.append(CL_UP)
FIX = pd.concat(parts, ignore_index=True).dropna(subset=["date", "league", "home", "away"])
FIX = FIX[FIX["date"] >= pd.Timestamp.today().normalize()].drop_duplicates(["date", "home", "away"]).reset_index(drop=True)

new_teams = sum((r.home not in ELO) or (r.away not in ELO) for r in FIX.itertuples())
X = pd.DataFrame([build(r.home, r.away, r.league, r._asdict()) for r in FIX.itertuples()])[FEATS]
P = m1f.predict_proba(X); pb = m2f.predict_proba(X)[:, 1]

FIXTURES = [{"date": str(r.date.date()), "league": r.league, "home": r.home, "away": r.away,
             "ph": round(float(P[i, 0]), 3), "pd": round(float(P[i, 1]), 3), "pa": round(float(P[i, 2]), 3),
             "p": round(float(pb[i]), 3)} for i, r in enumerate(FIX.itertuples())]
OUT = pd.DataFrame(FIXTURES)
OUT.to_csv(f"{SAVE_DIR}/nexus_predictions.csv", index=False)
print(len(OUT), "upcoming fixtures |", OUT["league"].nunique(), "competitions |", new_teams, "involve a team with no history")
print(OUT.sort_values("date").head(25).to_string(index=False))